<a href="https://colab.research.google.com/github/busraparlakk/Auto-MPG/blob/main/btk_rag_mimarisi_eklenmi%C5%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ortam kurulumu

In [1]:
!pip install transformers peft torch accelerate bitsandbytes
!pip install langchain langchain-community
!pip install sentence-transformers networkx
!pip install newspaper3k feedparser yfinance
!pip install huggingface_hub

modeli yükle (mevcut LoRA adapter ile)

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL = "unsloth/qwen2.5-7b-instruct-bnb-4bit"
ADAPTER    = "busraaparlak/qwen25-borsa-djia"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, ADAPTER)
model.eval()
print("✅ Model hazır")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model hazır


haber ve piyasa verisi çekici (Retrieval katmanı)

In [3]:
import yfinance as yf
import feedparser
import networkx as nx
from datetime import datetime, timedelta

# ── Piyasa verisi ──────────────────────────────────────────
def get_market_data(ticker="^DJI", days=7):
    end   = datetime.today()
    start = end - timedelta(days=days)
    df = yf.download(ticker, start=start, end=end, progress=False)
    df["RSI"] = compute_rsi(df["Close"])
    last = df.iloc[-1]
    return {
        "close"  : round(float(last["Close"]), 2),
        "change" : round(float(last["Close"] - df.iloc[-2]["Close"]), 2),
        "rsi"    : round(float(last["RSI"]), 2),
        "volume" : int(last["Volume"]),
    }

def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

# ── Haber çekici ───────────────────────────────────────────
RSS_FEEDS = [
    "https://feeds.finance.yahoo.com/rss/2.0/headline?s=^DJI&region=US&lang=en-US",
    "https://www.cnbc.com/id/20910258/device/rss/rss.html",   # CNBC Markets
]

def get_news(max_items=10):
    articles = []
    for url in RSS_FEEDS:
        feed = feedparser.parse(url)
        for entry in feed.entries[:max_items]:
            articles.append({
                "title"  : entry.get("title", ""),
                "summary": entry.get("summary", "")[:300],
                "date"   : entry.get("published", ""),
            })
    return articles[:max_items]

knowledge Graph inşası (Rag'un çekirdeği)

In [4]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def build_graph(articles):
    G = nx.DiGraph()
    for i, art in enumerate(articles):
        node_id = f"news_{i}"
        G.add_node(node_id,
                   title   = art["title"],
                   summary = art["summary"],
                   date    = art["date"])
    # Semantik benzerliğe göre kenar ekle
    texts = [a["title"] + " " + a["summary"] for a in articles]
    embs  = embedder.encode(texts, convert_to_tensor=True)
    sims  = util.cos_sim(embs, embs)
    for i in range(len(articles)):
        for j in range(len(articles)):
            if i != j and sims[i][j] > 0.45:
                G.add_edge(f"news_{i}", f"news_{j}",
                           weight=float(sims[i][j]))
    return G, embs, texts

def retrieve_relevant(query, G, embs, texts, top_k=5):
    q_emb   = embedder.encode(query, convert_to_tensor=True)
    scores  = util.cos_sim(q_emb, embs)[0]
    top_idx = scores.argsort(descending=True)[:top_k].tolist()
    results = []
    for idx in top_idx:
        node = f"news_{idx}"
        # Graf komşularını da dahil et (graph-aware)
        neighbors = list(G.successors(node))
        for nb in neighbors[:2]:
            data = G.nodes[nb]
            results.append(f"[İlgili] {data['title']}: {data['summary']}")
        data = G.nodes[node]
        results.append(f"[Ana] {data['title']}: {data['summary']}")
    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


rag pipeline ile çıktı üretimi

In [5]:
def rog_analyze(user_query: str, verbose=False):
    # 1. Veri çek
    market = get_market_data()
    news   = get_news()

    # 2. Graf kur ve ilgili haberleri bul
    G, embs, texts = build_graph(news)
    relevant       = retrieve_relevant(user_query, G, embs, texts, top_k=4)

    # 3. Bağlam oluştur
    context = f"""
📊 DJIA Piyasa Verisi ({datetime.today().strftime('%Y-%m-%d')}):
- Kapanış: {market['close']}  |  Değişim: {market['change']}
- RSI(14): {market['rsi']}    |  Hacim: {market['volume']:,}

📰 İlgili Haberler:
""" + "\n".join(f"• {r}" for r in relevant)

    if verbose:
        print(context)

    # 4. Prompt oluştur (Qwen chat formatı)
    messages = [
        {"role": "system", "content":
         "Sen uzman bir borsa analistisisin. "
         "Sana verilen güncel haber ve piyasa verilerini kullanarak "
         "DJIA hakkında detaylı, veriye dayalı analiz yap."},
        {"role": "user", "content":
         f"Bağlam:\n{context}\n\nSoru: {user_query}"}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 5. Üret
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response

# Test
result = rog_analyze("DJIA bugün ne yönde hareket eder? Haber akışına göre yorumla.")
print(result)

/tmp/ipykernel_19960/1772134981.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start, end=end, progress=False)
/tmp/ipykernel_19960/1772134981.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "close"  : round(float(last["Close"]), 2),
/tmp/ipykernel_19960/1772134981.py:15: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "change" : round(float(last["Close"] - df.iloc[-2]["Close"]), 2),
/tmp/ipykernel_19960/1772134981.py:16: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "rsi"    : round(float(last["RSI"]), 2),
/tmp/ipykernel_19960/1772134981.py:17: FutureWarning: Calling int on a single element Series is deprecated and will 

DJIA'nın bu haberi analiz et. Güçlü pozitif piyasa hareketi gözlemlenmiştir. Günlük değişim: %0.77, hacim: 104,661,401. Günlük RSI: NaN. Günlük AŞAĞI (down)


hugging Face Space olarak yayınla (Gradio)

In [6]:
# app.py — HF Space'e yüklenecek dosya
import gradio as gr

def analyze(query):
    return rog_analyze(query)

demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(label="Sorunuz", placeholder="DJIA bugün ne yapar?"),
    outputs=gr.Textbox(label="Analiz", lines=15),
    title="DJIA Borsa Analisti (RoG + Qwen2.5)",
    description="Güncel haberler + piyasa verisi + LoRA fine-tuned model",
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76a90e5fba97e0156f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
